# Garbage Image Classification using Transfer Learning

In [11]:
import warnings
warnings.filterwarnings('ignore')

import os
import random
import shutil
from pathlib import Path
import tensorflow as tf
import numpy as np
import pathlib
import matplotlib.pyplot as plt
import seaborn as sns
import mlflow
import dagshub
from dotenv import load_dotenv
import json
from tensorflow.keras.preprocessing import image_dataset_from_directory
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.layers.experimental.preprocessing import RandomZoom
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Rescaling
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import classification_report, confusion_matrix

### Counting the number of images present for each label

In [12]:
parent_folder = "../Datasets/12_Garbage_Class"
image_extensions = ('.jpg', '.jpeg', '.png')

data_dir = pathlib.Path(parent_folder)

for folder_name in os.listdir(parent_folder):
    img_count = len(list(data_dir.glob(f'{folder_name}/*')))
    print(f'{folder_name}: {img_count} images')

battery: 945 images
biological: 985 images
brown-glass: 607 images
cardboard: 891 images
clothes: 5325 images
green-glass: 629 images
metal: 769 images
paper: 1050 images
plastic: 865 images
shoes: 1977 images
trash: 697 images
white-glass: 775 images


# Train, Validation and Test Splitting

In [13]:
"""def split_dataset(parent_folder):

    # percentages
    train_ratio = 0.7
    val_ratio = 0.15
    test_ratio = 0.15

    # get base folder (Datasets/)
    base_folder = os.path.dirname(parent_folder)

    train_path = os.path.join(base_folder, "Train")
    val_path = os.path.join(base_folder, "Val")
    test_path = os.path.join(base_folder, "Test")

    # create main folders
    os.makedirs(train_path, exist_ok=True)
    os.makedirs(val_path, exist_ok=True)
    os.makedirs(test_path, exist_ok=True)

    # loop through each class
    for label in os.listdir(parent_folder):

        label_path = os.path.join(parent_folder, label)

        if not os.path.isdir(label_path):
            continue

        print("Processing:", label)

        images = os.listdir(label_path)

        # remove hidden files if any
        images = [img for img in images if not img.startswith('.')]

        random.shuffle(images)

        total = len(images)

        # calculate split counts
        train_count = int(total * train_ratio)
        val_count = int(total * val_ratio)

        # split images
        train_images = images[:train_count]
        val_images = images[train_count:train_count + val_count]
        test_images = images[train_count + val_count:]

        # create label folders
        train_label = os.path.join(train_path, label)
        val_label = os.path.join(val_path, label)
        test_label = os.path.join(test_path, label)

        os.makedirs(train_label, exist_ok=True)
        os.makedirs(val_label, exist_ok=True)
        os.makedirs(test_label, exist_ok=True)

        # copy files
        for img in train_images:
            shutil.copy(os.path.join(label_path, img), os.path.join(train_label, img))

        for img in val_images:
            shutil.copy(os.path.join(label_path, img), os.path.join(val_label, img))

        for img in test_images:
            shutil.copy(os.path.join(label_path, img), os.path.join(test_label, img))

        print(f"{label} → Total:{total} | Train:{len(train_images)}, Val:{len(val_images)}, Test:{len(test_images)}")

    print("\nDone!")"""

'def split_dataset(parent_folder):\n\n    # percentages\n    train_ratio = 0.7\n    val_ratio = 0.15\n    test_ratio = 0.15\n\n    # get base folder (Datasets/)\n    base_folder = os.path.dirname(parent_folder)\n\n    train_path = os.path.join(base_folder, "Train")\n    val_path = os.path.join(base_folder, "Val")\n    test_path = os.path.join(base_folder, "Test")\n\n    # create main folders\n    os.makedirs(train_path, exist_ok=True)\n    os.makedirs(val_path, exist_ok=True)\n    os.makedirs(test_path, exist_ok=True)\n\n    # loop through each class\n    for label in os.listdir(parent_folder):\n\n        label_path = os.path.join(parent_folder, label)\n\n        if not os.path.isdir(label_path):\n            continue\n\n        print("Processing:", label)\n\n        images = os.listdir(label_path)\n\n        # remove hidden files if any\n        images = [img for img in images if not img.startswith(\'.\')]\n\n        random.shuffle(images)\n\n        total = len(images)\n\n        # c

In [14]:
"""parent_folder = "../Datasets/12_Garbage_Class"
split_dataset(parent_folder)"""

'parent_folder = "../Datasets/12_Garbage_Class"\nsplit_dataset(parent_folder)'

# Data Preparation for training, evaluation and testing

In [15]:
train_path = "../Datasets/12_Train"
val_path = "../Datasets/12_Val"
test_path = "../Datasets/12_Test"

IMAGE_SIZE = (180, 180)
BATCH_SIZE = 32
SEED = 123

train_ds = image_dataset_from_directory(
  train_path,
  image_size=IMAGE_SIZE,
  batch_size=BATCH_SIZE,
  seed=SEED,
  label_mode='categorical'
)

val_ds = image_dataset_from_directory(
  val_path,
  image_size=IMAGE_SIZE,
  batch_size=BATCH_SIZE,
  seed=SEED,
  label_mode='categorical'
)

test_ds = image_dataset_from_directory(
    test_path,
    image_size=IMAGE_SIZE,
    batch_size = BATCH_SIZE,
    shuffle=False,
    label_mode = None
)

test_ds_with_labels = image_dataset_from_directory(
    test_path,
    image_size=IMAGE_SIZE,
    batch_size = BATCH_SIZE,
    shuffle=False,
    label_mode='categorical'
)

# For model performance evaluation
y_test = np.concatenate([y for x, y in test_ds_with_labels])
y_test_labels = np.argmax(y_test, axis=1)

class_names = train_ds.class_names
test_class_names = test_ds_with_labels.class_names


train_ds = train_ds.prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.prefetch(tf.data.AUTOTUNE)

"""train_ds = train_ds.apply(
  tf.data.experimental.ignore_errors()
)"""


Found 10854 files belonging to 12 classes.
Found 2321 files belonging to 12 classes.
Found 2340 files belonging to 1 classes.
Found 2340 files belonging to 12 classes.


'train_ds = train_ds.apply(\n  tf.data.experimental.ignore_errors()\n)'

### Saving Class Labels for Streamlit app plugin

In [16]:
with open("../app/data/class_names.json", 'w') as f:
    json.dump(class_names, f)

# CNN Model Pipeline

In [17]:
num_classes = len(class_names)

cnn_params = {
    "model_name": "Convolutional Neural Network",
    "input_shape": (180, 180, 3),
    "conv1_filters": 512,
    "conv2_filters": 128,
    #"conv3_filters": 64,
    "kernel_size": 3,
    "dense_units": 128,
    "optimizer": "adam",
    "loss": 'categorical_crossentropy',
    "epochs": 30,
    "early_stopping_patience": 15
}

cnn_model = Sequential([
    Rescaling(1./255, input_shape=cnn_params['input_shape']),
    Conv2D(cnn_params['conv1_filters'], cnn_params['kernel_size'], padding='same', activation='relu'),
    MaxPooling2D(),
    Conv2D(cnn_params['conv2_filters'], cnn_params['kernel_size'], padding='same', activation='relu'),
    MaxPooling2D(),
    #Conv2D(cnn_params['conv3_filters'], cnn_params['kernel_size'], padding='same', activation='relu'),
    #MaxPooling2D(),
    Flatten(),
    Dense(cnn_params['dense_units'], activation='relu'),
    Dense(num_classes, activation='softmax')
])

cnn_tags = {
    "framework": "tensorflow",
    "model_type": "Convolutional Neural Network"
}

# VGG16 Model Pipeline

In [18]:
vgg16_params ={
    "model_name":"VGG16",
    "input_shape": (180, 180, 3),
    "dense_units": 512,
    "optimizer": Adam(lr=0.001),
    "loss": 'categorical_crossentropy',
    "epochs": 10,
    "early_stopping_patience": 15
}

vgg16_model = Sequential()

pretrained_model = tf.keras.applications.VGG16(
    include_top=False,
    input_shape=vgg16_params['input_shape'],
    pooling='avg',
    classes=len(class_names),
    weights='imagenet'
)

for layer in pretrained_model.layers:
    layer.trainable=False

vgg16_model.add(pretrained_model)
vgg16_model.add(Flatten())
vgg16_model.add(Dense(vgg16_params['dense_units'], activation='relu'))
vgg16_model.add(Dense(len(class_names), activation='softmax'))

vgg16_tags = {
    "framework":"tensorflow",
    "model_type":'VGG16'
}

# VGG19 Model Pipeline

In [19]:
vgg19_params ={
    "model_name":"VGG19",
    "input_shape": (180, 180, 3),
    "dense_units": 512,
    "optimizer": Adam(lr=0.001),
    "loss": 'categorical_crossentropy',
    "epochs": 10,
    "early_stopping_patience": 15
}

vgg19_model = Sequential()

pretrained_model = tf.keras.applications.VGG19(
    include_top=False,
    input_shape=vgg19_params['input_shape'],
    pooling='avg',
    classes=len(class_names),
    weights='imagenet'
)

for layer in pretrained_model.layers:
    layer.trainable=False

vgg19_model.add(pretrained_model)
vgg19_model.add(Flatten())
vgg19_model.add(Dense(vgg19_params['dense_units'], activation='relu'))
vgg19_model.add(Dense(len(class_names), activation='softmax'))

vgg19_tags = {
    "framework":"tensorflow",
    "model_type":'VGG19'
}

# MobileNet Model Pipeline

In [20]:
MobileNet_params ={
    "model_name":"MobileNet",
    "input_shape": (180, 180, 3),
    "dense_units": 512,
    "optimizer": Adam(lr=0.001),
    "loss": 'categorical_crossentropy',
    "epochs": 10,
    "early_stopping_patience": 15
}

MobileNet_model = Sequential()

pretrained_model = tf.keras.applications.MobileNet(
    include_top=False,
    input_shape=MobileNet_params['input_shape'],
    pooling='avg',
    classes=len(class_names),
    weights='imagenet'
)

for layer in pretrained_model.layers:
    layer.trainable=False

MobileNet_model.add(pretrained_model)
MobileNet_model.add(Flatten())
MobileNet_model.add(Dense(MobileNet_params['dense_units'], activation='relu'))
MobileNet_model.add(Dense(len(class_names), activation='softmax'))

MobileNet_tags = {
    "framework":"tensorflow",
    "model_type":'MobileNet'
}

# MobileNetV2 Model Pipeline

In [21]:
MobileNetV2_params ={
    "model_name":"MobileNetV2",
    "input_shape": (180, 180, 3),
    "dense_units": 512,
    "optimizer": Adam(lr=0.001),
    "loss": 'categorical_crossentropy',
    "epochs": 10,
    "early_stopping_patience": 15
}

MobileNetV2_model = Sequential()

pretrained_model = tf.keras.applications.MobileNetV2(
    include_top=False,
    input_shape=MobileNetV2_params['input_shape'],
    pooling='avg',
    classes=len(class_names),
    weights='imagenet'
)

for layer in pretrained_model.layers:
    layer.trainable=False

MobileNetV2_model.add(pretrained_model)
MobileNetV2_model.add(Flatten())
MobileNetV2_model.add(Dense(MobileNetV2_params['dense_units'], activation='relu'))
MobileNetV2_model.add(Dense(len(class_names), activation='softmax'))

MobileNetV2_tags = {
    "framework":"tensorflow",
    "model_type":'MobileNetV2'
}

# ResNet50 Model Pipeline

In [22]:
resnet_params ={
    "model_name":"ResNet50",
    "input_shape": (180, 180, 3),
    "dense_units": 512,
    "optimizer": Adam(lr=0.001),
    "loss": 'categorical_crossentropy',
    "epochs": 10,
    "early_stopping_patience": 15
}

resnet_model = Sequential()

pretrained_model = tf.keras.applications.ResNet50(
    include_top=False,
    input_shape=resnet_params['input_shape'],
    pooling='avg',
    classes=len(class_names),
    weights='imagenet'
)

for layer in pretrained_model.layers:
    layer.trainable=False

resnet_model.add(pretrained_model)
resnet_model.add(Flatten())
resnet_model.add(Dense(resnet_params['dense_units'], activation='relu'))
resnet_model.add(Dense(len(class_names), activation='softmax'))

resnet50_tags = {
    "framework":"tensorflow",
    "model_type":'ResNet50'
}

# Model Training

In [ ]:
models = [
    #(
    #    "Convolutional Neural Network",
    #    cnn_model,
    #    train_ds,
    #    val_ds,
    #    test_ds,
    #    y_test,
    #    cnn_params,
    #    cnn_tags
    #),
    (
        "VGG16",
        vgg16_model,
        train_ds,
        val_ds,
        test_ds,
        y_test,
        vgg16_params,
        vgg16_tags
    ),
    (
        "VGG19",
        vgg19_model,
        train_ds,
        val_ds,
        test_ds,
        y_test,
        vgg19_params,
        vgg19_tags
    ),
    (
        "MobileNet",
        MobileNet_model,
        train_ds,
        val_ds,
        test_ds,
        y_test,
        MobileNet_params,
        MobileNet_tags
    ),
    (
        "MobileNetV2",
        MobileNetV2_model,
        train_ds,
        val_ds,
        test_ds,
        y_test,
        MobileNetV2_params,
        MobileNetV2_tags
    ),
    (
        "ResNet50",
        resnet_model,
        train_ds,
        val_ds,
        test_ds,
        y_test,
        resnet_params,
        resnet50_tags
    )
]

In [ ]:
model_reports = []
model_history = []

for model_name, model, train_data, val_data, test_data, y_true, model_param, model_tag in models:

  train_ds = train_data
  val_ds = val_data
  test_ds = test_data
  y_test = y_true

  model.compile(
      optimizer=model_param['optimizer'],
      loss=model_param['loss'],
      metrics=['accuracy']
  )

  print(f"{model_name} Compiled")

  early_stop = EarlyStopping(
    monitor='val_loss',
    patience=model_param['early_stopping_patience'],
    restore_best_weights=True
  )

  print(f"{model_name} training started")

  history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=model_param['epochs'],
    callbacks=[early_stop]
  )

  print(f"{model_name} Trained Successfully!")

  model_history.append(history)

  predictions = model.predict(test_ds)
  y_pred = np.argmax(predictions, axis=1)

  # Classification Report
  report = classification_report(y_test_labels, y_pred, target_names=test_class_names, output_dict=True)
  model_reports.append(report)

  # Confusion Matrix
  cm = confusion_matrix(y_test_labels, y_pred, labels=np.arange(len(class_names)))
  
  save_dir = os.path.join("Images", "Confusion Matrix")
  os.makedirs(save_dir, exist_ok=True)
  file_path = os.path.join(save_dir, f"{model_name}.png")

  plt.figure(figsize=(8, 6))
  sns.heatmap(
    cm, 
    annot=True, 
    fmt='d', 
    cmap='Blues',
    xticklabels=class_names,
    yticklabels=class_names
  )
  plt.title(f"{model_name} Confusion Matrix")
  plt.xlabel("Predicted")
  plt.ylabel("Actual")

  # Save (overwrite)
  plt.savefig(file_path, bbox_inches='tight')
  plt.close()
  print(f"{model_name} Confusion Matrix saved at {file_path}")

  print(f"{model_name} Performance Reports and history added!")

VGG16 Compiled
VGG16 training started
Epoch 1/10
340/340 [==============================] - 55s 136ms/step - loss: 0.9534 - accuracy: 0.8252 - val_loss: 0.4533 - val_accuracy: 0.8875
Epoch 2/10
340/340 [==============================] - 30s 87ms/step - loss: 0.2087 - accuracy: 0.9345 - val_loss: 0.3616 - val_accuracy: 0.9035
Epoch 3/10
340/340 [==============================] - 28s 83ms/step - loss: 0.0956 - accuracy: 0.9703 - val_loss: 0.4630 - val_accuracy: 0.8992
Epoch 4/10
340/340 [==============================] - 28s 82ms/step - loss: 0.0670 - accuracy: 0.9803 - val_loss: 0.4143 - val_accuracy: 0.9078
Epoch 5/10
340/340 [==============================] - 27s 80ms/step - loss: 0.0455 - accuracy: 0.9872 - val_loss: 0.3779 - val_accuracy: 0.9203
Epoch 6/10
340/340 [==============================] - 27s 81ms/step - loss: 0.0362 - accuracy: 0.9894 - val_loss: 0.4968 - val_accuracy: 0.9026
Epoch 7/10
340/340 [==============================] - 28s 82ms/step - loss: 0.1028 - accuracy: 0.

# Model Tracking in Dagshub

In [25]:
dagshub.init(repo_owner='JS-Tharun', repo_name='Garbage-Image-Classifier', mlflow=True)

load_dotenv()

os.environ['MLFLOW_TRACKING_USERNAME'] = f"{os.getenv('MLFLOW_USERNAME')}"
os.environ['MLFLOW_TRACKING_PASSWORD'] = f"{os.getenv('MLFLOW_PASSWORD')}"



mlflow.set_tracking_uri(os.environ['MLFLOW_TRACKING_URI'])
mlflow.set_experiment(os.environ["MLFLOW_EXPERIMENT_NAME"])

for i, element in enumerate(models):
  model_name = element[0]
  model = element[1]
  train_ds = element[2]
  val_ds = element[3]
  test_ds = element[4]
  y_test = element[5]
  param = element[6]
  tag = element[7]

  model_report = model_reports[i]
  history = model_history[i]


  with mlflow.start_run(run_name=model_name):

    #--------------------------------------
    # Data Pipeline logging
    #--------------------------------------

    mlflow.set_tags(tag)

    mlflow.log_param("train_data_path", train_path)
    mlflow.log_param("val_data_path", val_path)
    mlflow.log_param("Seed", SEED)
    mlflow.log_param("image_height", IMAGE_SIZE[0])
    mlflow.log_param("image_width", IMAGE_SIZE[1])
    mlflow.log_param("batch_size", BATCH_SIZE)
    mlflow.log_param("class_names", class_names)
    mlflow.log_params(param)

    print(f"{model_name} parameters logged!")

    #-----------------------------------------------------
    # Model training and validation performance tracking
    #-----------------------------------------------------

    for epoch in range(len(history.history['loss'])):
      mlflow.log_metric("train_loss", history.history['loss'][epoch], step=epoch)
      mlflow.log_metric("train_accuracy", history.history['accuracy'][epoch], step=epoch)
      mlflow.log_metric("val_loss", history.history['val_loss'][epoch], step=epoch)
      mlflow.log_metric("val_accuracy", history.history['val_accuracy'][epoch], step=epoch)

    #------------------------------------------------------
    # Log Model
    #------------------------------------------------------

    model_info = mlflow.tensorflow.log_model(
      model, 
      artifact_path=model_name
    )

    print(f"{model_name} model logged!")

    #------------------------------------------------------
    # Model testing performance tracking
    #------------------------------------------------------

    # ✅ Log best epoch metrics explicitly (for clean experiment table comparison)
    best_epoch = np.argmin(history.history['val_loss'])
    mlflow.log_metric("best_epoch", best_epoch)
    mlflow.log_metric("best_val_loss", history.history['val_loss'][best_epoch])
    mlflow.log_metric("best_val_accuracy", history.history['val_accuracy'][best_epoch])
    mlflow.log_metric("best_train_accuracy", history.history['accuracy'][best_epoch])

    # ✅ Link test metrics to the LoggedModel using model_id
    mlflow_metrics = {}
    for label, metrics in model_report.items():
      if isinstance(metrics, dict):
        for metric_name, value in metrics.items():
          if metric_name != "support":
            mlflow_metrics[f"{label}_{metric_name}"] = float(value)
      else:
        mlflow_metrics[label] = float(metrics)

    # Pass model_id to link metrics to the model in the Models tab
    mlflow.log_metrics(mlflow_metrics, model_id=model_info.model_id)

    print(f"{model_name} metrics logged")


    #--------------------------------------------------------
    # Logging Confusion Matrix Artifact
    #--------------------------------------------------------

    cm_path = os.path.join("Images", "Confusion Matrix", f"{model_name}.png")
    if os.path.exists(cm_path):
      mlflow.log_artifact(cm_path, artifact_path="confusion_matrices")
      print(f"{model_name} confusion matrix logged to MLflow!")
    else:
        print(f"{model_name} confusion matrix not found!")


Accessing as workspacejst

Initialized MLflow to track repo "JS-Tharun/Garbage-Image-Classifier"

Repository JS-Tharun/Garbage-Image-Classifier initialized!

VGG16 parameters logged!


2026/04/20 12:57:17 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/20 12:57:26 WARNING mlflow.tensorflow: You are saving a TensorFlow Core model or Keras model without a signature. Inference with mlflow.pyfunc.spark_udf() will not work unless the model's pyfunc representation accepts pandas DataFrames as inference inputs.


...

MobileNetV2 model logged!
MobileNetV2 metrics logged
MobileNetV2 confusion matrix logged to MLflow!
🏃 View run MobileNetV2 at: https://dagshub.com/JS-Tharun/Garbage-Image-Classifier.mlflow/#/experiments/1/runs/5380d34151064b4b977c759ff811f5a9
🧪 View experiment at: https://dagshub.com/JS-Tharun/Garbage-Image-Classifier.mlflow/#/experiments/1
ResNet50 parameters logged!


2026/04/20 13:04:05 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/20 13:04:15 WARNING mlflow.tensorflow: You are saving a TensorFlow Core model or Keras model without a signature. Inference with mlflow.pyfunc.spark_udf() will not work unless the model's pyfunc representation accepts pandas DataFrames as inference inputs.


INFO:tensorflow:Assets written to: C:\Users\jstha\AppData\Local\Temp\tmpiug2tnnb\model\data\model\assets


INFO:tensorflow:Assets written to: C:\Users\jstha\AppData\Local\Temp\tmpiug2tnnb\model\data\model\assets


ResNet50 model logged!
ResNet50 metrics logged
ResNet50 confusion matrix logged to MLflow!
🏃 View run ResNet50 at: https://dagshub.com/JS-Tharun/Garbage-Image-Classifier.mlflow/#/experiments/1/runs/235e606ab02f4b6bbb9a57b29ff3ee6c
🧪 View experiment at: https://dagshub.com/JS-Tharun/Garbage-Image-Classifier.mlflow/#/experiments/1
